In [27]:
import pandas as pd
import re
from numerizer import numerize



# ========= Load Excel =========
def load_capacity_column(file_path):
    df = pd.read_excel(file_path)
    df = df[["capacity", "product"]].copy()
    df["capacity"] = df["capacity"].fillna("")
    return df

# ========= Scale Mapping =========
SCALE_MAP = {
    # Thousand
    "thousand": 1e3, "thousands": 1e3,
    "k": 1e3, "K": 1e3,
    "kilo": 1e3, "kilos": 1e3,
    
    # Million
    "million": 1e6, "millions": 1e6,
    "m": 1e6, "M": 1e6,
    "mn": 1e6, "MN": 1e6,
    "mln": 1e6, "MLN": 1e6,
    
    # Billion
    "billion": 1e9, "billions": 1e9,
    "b": 1e9, "B": 1e9,
    "bn": 1e9, "BN": 1e9,
    "bln": 1e9, "BLN": 1e9,
    
    # Trillion
    "trillion": 1e12, "trillions": 1e12,
    "t": 1e12, "T": 1e12,
    "tn": 1e12, "TN": 1e12,
    "trn": 1e12, "TRN": 1e12
}


# ========= Time Mapping =========
TIME_MAP = {
    "yearly": "per year", "/year": "per year","/yr": "per year", "annually": "per year", "per annum": "per year", "1 year": "per year",
    "per year": "per year", "a year": "per year",
    "monthly": "per month", "per month": "per month", "/month": "per month", "a month": "per month",
    "weekly": "per week", "per week": "per week", "/week": "per week",
    "daily": "per day", "per day": "per day", "/day": "per day", "a day": "per day",
    "hourly": "per hour", "per hour": "per hour", "/hour": "per hour", "an hour": "per hour", "-hours": "per hour",  "-hour": "per hour", "- hour": "per hour", "hour":"per hour", "- hours": "per hour", " -hour": "per hour"
}

# ========= Metric Mapping =========
METRIC_MAP = {
    "gwh": "gigawatt hour","gigawatt": "gigawatt", "gigawatts hours":"gigawatt hour","gigawatts-hours":"gigawatt hour", "gigawatt-hours":"gigawatt hour","gigawatts-hour":"gigawatt hour","gigawatt hours": "gigawatt hour","mwh": "megawatt hour", "kwh": "kilowatt hour", "twh": "terawatt hour", "wh": "watt hour",
    "mw": "megawatt", "kw": "kilowatt", "tw": "terawatt",
    "t": "tonne","tonne": "tonne", "tonnes": "tonne", "tons": "tonne", "ton": "tonne", "mt": "megatonne", "kt": "kilotonne",
    "kg": "kilogram", "g": "gram", "units": "unit", "unit": "unit"
}

# ========= Regex Patterns =========
REGEX_PATTERNS = {
    "range": r"^([0-9,.]+)\s*(?:to|-|–|—)\s*([0-9,.]+)\s+(.*)",
    
    "approx_scaled": r"^(almost|nearly|around|about|approximately|approx\.?|over|under|more than|less than)\s+([0-9,.]+)\s+(thousand|million|billion|trillion)\b\s*(.*)",
    
    "approx": r"^(almost|nearly|around|about|approximately|approx\.?|over|under|more than|less than|up to)\s+([0-9,.]+)\s+(.*)",
    
    "fractional_half": r"^(half|a half|one half|one and a half| half a)\s+(thousand|million|billion|trillion)\b\s*(.*)",
    
    "compound_word_scale": r"^([a-z\s-]+)\s+(thousand|million|billion|trillion)\b\s*(.*)",
    
    "num_with_scale": r"^([0-9,.]+)\s+(thousand|million|billion|trillion)\b\s*(.*)",
    
    "word_with_scale": r"^([a-z\s-]+)\s+(thousand|million|billion|trillion)\b\s*(.*)",
    
    "plain_numeric": r"^([0-9,.]+)\s*([a-zA-Z/]+.*)"
}

# ========= Capacity Info Extraction =========

def extract_capacity_info(text):
    if not isinstance(text, str):
        return None, None, None
    text = text.strip()

    # Case 0: Range (e.g., 1,000–2,000 tonnes)
    match = re.match(REGEX_PATTERNS["range"], text, re.IGNORECASE)
    if match:
        try:
            val1 = float(match.group(1).replace(",", ""))
            val2 = float(match.group(2).replace(",", ""))
            remaining = match.group(3).strip()
            return [val1, val2], None, remaining
        except:
            pass

    # Case 0.5: Approximate with scale (e.g., about 1 million)
    match = re.match(REGEX_PATTERNS["approx_scaled"], text, re.IGNORECASE)
    if match:
        value = float(match.group(2).replace(",", ""))
        scale = SCALE_MAP.get(match.group(3).lower())
        remaining = match.group(4).strip()
        return value, scale, remaining

    # Case 0.6: Approximate without scale (e.g., about 2000)
    match = re.match(REGEX_PATTERNS["approx"], text, re.IGNORECASE)
    if match:
        value = float(match.group(2).replace(",", ""))
        remaining = match.group(3).strip()
        return value, None, remaining

    # Case 1: Numeric with scale (e.g., 2 million)
    match = re.match(REGEX_PATTERNS["num_with_scale"], text, re.IGNORECASE)
    if match:
        value = float(match.group(1).replace(",", ""))
        scale = SCALE_MAP.get(match.group(2).lower())
        remaining = match.group(3).strip()
        return value, scale, remaining

    # Case 2: Word with scale (e.g., one million)
    match = re.match(REGEX_PATTERNS["word_with_scale"], text, re.IGNORECASE)
    if match:
        try:
            value = w2n.word_to_num(match.group(1).strip())
            scale = SCALE_MAP.get(match.group(2).lower())
            remaining = match.group(3).strip()
            return value, scale, remaining
        except:
            pass
    # Case: number + k (e.g., 5k, 100K) treated as scale if 'k' is alone
    match_k_scale = re.match(r"^([0-9,.]+)\s*[kK]\b\s*(.*)", text)
    if match_k_scale:
        try:
            value = float(match_k_scale.group(1).replace(",", ""))
            remaining = match_k_scale.group(2).strip()
            scale = SCALE_MAP.get("k")
            return value, scale, remaining
        except:
            pass

    # Case 3: Numeric stuck to unit (e.g., 20GWh, 40t, 1000MW/year)
    match = re.match(REGEX_PATTERNS["plain_numeric"], text, re.IGNORECASE)
    if match:
        value = float(match.group(1).replace(",", ""))
        remaining = match.group(2).strip()
        return value, None, remaining

    # Fallback: Clean and parse with numerizer (handles: "up to one million heaters", "around three billion")
    try:
        # Remove leading modifier words
        modifiers = ["up to", "about", "around", "approximately", "approx", "nearly", "almost", "more than", "less than", "over", "under"]
        text_lower = text.lower().strip()
        for mod in modifiers:
            if text_lower.startswith(mod):
                text_lower = text_lower[len(mod):].strip()
                break

        numerized = numerize(text_lower)
        match = re.match(r"^([0-9,.]+)\s*(thousand|million|billion|trillion)?\b\s*(.*)", numerized, re.IGNORECASE)
        if match:
            value = float(match.group(1).replace(",", ""))
            scale_str = match.group(2)
            remaining = match.group(3).strip()
            scale = SCALE_MAP.get(scale_str.lower()) if scale_str else None
            return value, scale, remaining
    except:
        pass



    return None, None, text

# ========= Time Unit Extraction =========
def extract_normalized_time_unit(text):
    if not isinstance(text, str):
        return None, text

    text_lower = text.lower()
    for pattern, replacement in TIME_MAP.items():
        if re.search(rf"(?<![a-zA-Z0-9]){re.escape(pattern)}(?![a-zA-Z0-9])|[-/]{re.escape(pattern)}", text_lower):
            cleaned = re.sub(rf"(?<![a-zA-Z0-9]){re.escape(pattern)}(?![a-zA-Z0-9])|[-/]{re.escape(pattern)}", "", text, flags=re.IGNORECASE).strip()
            return replacement, cleaned
    return None, text


# ========= Metric Unit Extraction =========
def extract_normalized_metric_unit(text):
    if not isinstance(text, str):
        return None, text

    text_lower = text.lower()
    for pattern, replacement in METRIC_MAP.items():
        # match units stuck to numbers or attached with hyphen/slash
        if re.search(rf"(?<![a-zA-Z0-9]){re.escape(pattern)}(?![a-zA-Z0-9])|[-/]{re.escape(pattern)}", text_lower):
            cleaned = re.sub(rf"(?<![a-zA-Z0-9]){re.escape(pattern)}(?![a-zA-Z0-9])|[-/]{re.escape(pattern)}", "", text, flags=re.IGNORECASE).strip()
            return replacement, cleaned
    return None, text


# ========= Pipeline Execution =========
def run_extraction_pipeline(file_path):
    df = load_capacity_column(file_path)
    df[["value", "scale", "text"]] = df["capacity"].apply(lambda x: pd.Series(extract_capacity_info(x)))
    df[["metric", "text"]] = df["text"].apply(lambda x: pd.Series(extract_normalized_metric_unit(x)))
    df[["time", "text"]] = df["text"].apply(lambda x: pd.Series(extract_normalized_time_unit(x)))
    return df

# ========= Main Run Block =========
if __name__ == "__main__":
    file_path = 'C:/Users/marie.juge/OneDrive - Bruegel/Desktop/tuone/tuone_v6/reconcile/storage/output/clean_output_ben.xlsx'
    df_result = run_extraction_pipeline(file_path)
    df_text = df_result["text"].unique()
    print(df_result.head(10))


                            capacity                   product     value  \
0                                                      battery      None   
1                60,000 EVs per year            ioniq electric   60000.0   
2                    40 GWh per year          li ion batteries      40.0   
3      1,000 electric buses per year            electric buses    1000.0   
4               5,000 units per year            electric buses    5000.0   
5  10,000 delivery vehicles per year         streetscooter van   10000.0   
6                                        compact electric cars      None   
7                 100 units per year              electric bus     100.0   
8             30,000 tonnes per year         lithium carbonate   30000.0   
9             400,000 units per year  electric and hybrid cars  400000.0   

   scale               text         metric      time  
0    NaN                              None      None  
1    NaN                EVs           None  per year 

To do:
- check k 200k units 200kW
- check for -hour